# ensemble model experiments

systematic experimentation script for ensemble meta-learner tuning.
tests different combinations of base models and hyperparameter settings
to find optimal ensemble configurations.

In [9]:
import os
import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, Tuple, List, Any, Optional
from dataclasses import dataclass, asdict
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


## configuration

In [10]:
@dataclass
class ExperimentConfig:
    predictions_file: str = "base_predictions.npz"
    results_dir: str = "experiment_results"
    seed: int = 8
    n_splits: int = 5
    verbose: bool = True

config = ExperimentConfig()
os.makedirs(config.results_dir, exist_ok=True)

## load predictions

In [11]:
def load_predictions(npz_file: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[str]]:
    """load pre-computed predictions from NPZ file."""
    if not Path(npz_file).exists():
        raise FileNotFoundError(f"File not found: {npz_file}\nRun precompute_predictions.py first!")

    data = np.load(npz_file, allow_pickle=True)

    X_stacked = data['X_stacked']
    y_encoded = data['y']
    y_original = data['y_original']
    species_classes = data['species_classes']
    model_names = data['model_names']

    return X_stacked, y_encoded, y_original, list(species_classes), list(model_names)

## meta-learner classes

In [12]:
class RFMetaLearner:
    """random forest meta-learner with configurable hyperparameters."""
    def __init__(self, n_estimators=200, max_depth=15, min_samples_leaf=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.model = None
        self.le = LabelEncoder()

    def fit(self, X_meta: np.ndarray, y: np.ndarray):
        self.le.fit(y)
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            random_state=config.seed,
            n_jobs=-1
        )
        self.model.fit(X_meta, y)

    def predict_proba(self, X_meta: np.ndarray) -> np.ndarray:
        return self.model.predict_proba(X_meta)

    def predict(self, X_meta: np.ndarray) -> np.ndarray:
        return self.model.predict(X_meta)

In [13]:
class XGBMetaLearner:
    """xgboost meta-learner with configurable hyperparameters."""
    def __init__(self, max_depth=7, eta=0.1, subsample=0.8, num_boost_round=100):
        self.max_depth = max_depth
        self.eta = eta
        self.subsample = subsample
        self.num_boost_round = num_boost_round
        self.model = None
        self.le = LabelEncoder()
        self.classes_ = None

    def fit(self, X_meta: np.ndarray, y: np.ndarray) -> bool:
        self.le.fit(y)
        self.classes_ = self.le.classes_
        y_encoded = self.le.transform(y)

        dtrain = xgb.DMatrix(X_meta, label=y_encoded)
        params = {
            'objective': 'multi:softprob',
            'num_class': len(self.classes_),
            'max_depth': self.max_depth,
            'eta': self.eta,
            'subsample': self.subsample,
            'colsample_bytree': 0.8,
            'random_state': config.seed
        }
        self.model = xgb.train(params, dtrain, num_boost_round=self.num_boost_round, verbose_eval=False)
        return True

    def predict_proba(self, X_meta: np.ndarray) -> Optional[np.ndarray]:
        if self.model is None:
            return None
        dtest = xgb.DMatrix(X_meta)
        return self.model.predict(dtest)

    def predict(self, X_meta: np.ndarray) -> Optional[np.ndarray]:
        proba = self.predict_proba(X_meta)
        if proba is None:
            return None
        return self.classes_[np.argmax(proba, axis=1)]

In [14]:
class NNMetaLearner:
    """neural network meta-learner with configurable architecture."""
    def __init__(self, input_dim: int, num_classes: int, hidden_dims: Tuple[int, ...] = (256, 128, 64), dropout_rates: Tuple[float, ...] = (0.4, 0.3, 0.2)):
        self.le = LabelEncoder()
        self.num_classes = num_classes

        # build network with specified hidden dimensions and dropout rates
        layers = []
        prev_dim = input_dim
        for hidden_dim, dropout_rate in zip(hidden_dims, dropout_rates):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, num_classes))

        self.model = nn.Sequential(*layers).to(DEVICE)
        self.optimizer = None
        self.criterion = None

    def fit(self, X_meta: np.ndarray, y: np.ndarray, epochs: int = 100, batch_size: int = 32) -> bool:
        self.le.fit(y)
        y_encoded = self.le.transform(y)

        X_tensor = torch.FloatTensor(X_meta).to(DEVICE)
        y_tensor = torch.LongTensor(y_encoded).to(DEVICE)

        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)

        self.model.train()
        for _ in range(epochs):
            for X_batch, y_batch in loader:
                self.optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = self.criterion(outputs, y_batch)
                loss.backward()
                self.optimizer.step()

        return True

    def predict_proba(self, X_meta: np.ndarray) -> Optional[np.ndarray]:
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_meta).to(DEVICE)
            outputs = self.model(X_tensor)
            probs = torch.softmax(outputs, dim=1)
            return probs.cpu().numpy()

    def predict(self, X_meta: np.ndarray) -> Optional[np.ndarray]:
        proba = self.predict_proba(X_meta)
        if proba is None:
            return None
        return self.le.classes_[np.argmax(proba, axis=1)]

## experiment runner

In [15]:
def run_experiment(X_stacked: np.ndarray, y_encoded: np.ndarray, y_original: np.ndarray,
                   species_classes: List[str], model_names: List[str],
                   model_indices: List[int],
                   rf_config: Optional[Dict] = None,
                   xgb_config: Optional[Dict] = None,
                   nn_config: Optional[Dict] = None,
                   experiment_name: str = "") -> Dict[str, float]:
    """run single experiment with specified model combination and hyperparameters."""

    # default configs
    if rf_config is None:
        rf_config = {'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 2}
    if xgb_config is None:
        xgb_config = {'max_depth': 7, 'eta': 0.1, 'subsample': 0.8, 'num_boost_round': 100}
    if nn_config is None:
        nn_config = {'hidden_dims': (256, 128, 64), 'dropout_rates': (0.4, 0.3, 0.2), 'epochs': 100}

    # select features for this experiment
    num_classes = len(species_classes)
    selected_models = [model_names[i] for i in model_indices]

    # build feature indices for selected models
    feature_indices = []
    for idx in model_indices:
        start = idx * num_classes
        end = (idx + 1) * num_classes
        feature_indices.extend(range(start, end))

    X_selected = X_stacked[:, feature_indices]

    # create label encoder
    le = LabelEncoder()
    le.fit(y_encoded)

    # k-fold cross-validation
    skf = StratifiedKFold(n_splits=config.n_splits, shuffle=True, random_state=config.seed)
    fold_accs = []
    fold_f1s = []

    for train_idx, test_idx in skf.split(X_selected, y_encoded):
        X_train = X_selected[train_idx]
        X_test = X_selected[test_idx]
        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        # train meta-learners
        best_acc = 0

        # RF Meta
        rf_meta = RFMetaLearner(**rf_config)
        rf_meta.fit(X_train, y_train)
        rf_pred = rf_meta.predict(X_test)
        rf_acc = accuracy_score(y_test, rf_pred)
        best_acc = max(best_acc, rf_acc)

        # XGB Meta
        xgb_meta = XGBMetaLearner(**xgb_config)
        if xgb_meta.fit(X_train, y_train):
            xgb_pred = xgb_meta.predict(X_test)
            xgb_acc = accuracy_score(y_test, xgb_pred)
            best_acc = max(best_acc, xgb_acc)

        # NN Meta
        nn_meta = NNMetaLearner(X_train.shape[1], num_classes, **{k: v for k, v in nn_config.items() if k in ['hidden_dims', 'dropout_rates']})
        epochs = nn_config.get('epochs', 100)
        if nn_meta.fit(X_train, y_train, epochs=epochs):
            nn_pred = nn_meta.predict(X_test)
            nn_acc = accuracy_score(y_test, nn_pred)
            best_acc = max(best_acc, nn_acc)

        fold_accs.append(best_acc)
        fold_f1s.append(best_acc)

    return {
        'experiment_name': experiment_name,
        'models': '+'.join(selected_models),
        'n_models': len(selected_models),
        'n_features': X_selected.shape[1],
        'mean_accuracy': np.mean(fold_accs),
        'std_accuracy': np.std(fold_accs),
        'min_accuracy': np.min(fold_accs),
        'max_accuracy': np.max(fold_accs),
    }

## load data and run experiments

In [16]:
# load data
X_stacked, y_encoded, y_original, species_classes, model_names = load_predictions(config.predictions_file)

print(f"\n[info] loaded {len(X_stacked)} samples, {len(species_classes)} species")
print(f"[info] base models: {list(model_names)}")
print(f"[info] total features: {X_stacked.shape[1]}")


[info] loaded 651 samples, 57 species
[info] base models: [np.str_('rf'), np.str_('rb'), np.str_('stats'), np.str_('resnet'), np.str_('cnn')]
[info] total features: 285


### phase 1: model combinations

In [ ]:
results = []
num_models = len(model_names)

print(f"\n{'='*80}")
print("phase 1: testing model combinations")
print(f"{'='*80}\n")

# test all combinations from 2 to 5 models
for r in range(2, num_models + 1):
    print(f"[phase1] testing {r}-model combinations...")

    for combo in combinations(range(num_models), r):
        combo_names = [model_names[i] for i in combo]
        exp_name = f"combo_{'+'.join(combo_names)}"

        result = run_experiment(
            X_stacked, y_encoded, y_original, species_classes, model_names,
            list(combo),
            experiment_name=exp_name
        )
        results.append(result)

        print(f"  {exp_name:40s} acc: {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")


phase 1: testing model combinations

[phase1] testing 2-model combinations...


### phase 2: hyperparameter tuning

In [ ]:
print(f"\n{'='*80}")
print("phase 2: hyperparameter tuning (all 5 models)")
print(f"{'='*80}\n")

all_indices = list(range(num_models))

# RF hyperparameters
print("[phase2] testing random forest hyperparameters...")
for n_est in [100, 200, 300]:
    for max_d in [10, 15, 20]:
        exp_name = f"rf_n{n_est}_d{max_d}"
        result = run_experiment(
            X_stacked, y_encoded, y_original, species_classes, model_names,
            all_indices,
            rf_config={'n_estimators': n_est, 'max_depth': max_d, 'min_samples_leaf': 2},
            experiment_name=exp_name
        )
        results.append(result)
        print(f"  {exp_name:40s} acc: {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")

# XGB hyperparameters
print("[phase2] testing xgboost hyperparameters...")
for max_d in [5, 7, 10]:
    for eta in [0.05, 0.1, 0.2]:
        exp_name = f"xgb_d{max_d}_eta{eta}"
        result = run_experiment(
            X_stacked, y_encoded, y_original, species_classes, model_names,
            all_indices,
            xgb_config={'max_depth': max_d, 'eta': eta, 'subsample': 0.8, 'num_boost_round': 100},
            experiment_name=exp_name
        )
        results.append(result)
        print(f"  {exp_name:40s} acc: {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")

# NN hyperparameters
print("[phase2] testing neural network architectures...")
architectures = [
    {'hidden_dims': (128, 64), 'dropout_rates': (0.3, 0.2), 'epochs': 100},
    {'hidden_dims': (256, 128), 'dropout_rates': (0.4, 0.3), 'epochs': 100},
    {'hidden_dims': (512, 256, 128), 'dropout_rates': (0.5, 0.4, 0.3), 'epochs': 100},
    {'hidden_dims': (256, 128, 64, 32), 'dropout_rates': (0.4, 0.3, 0.2, 0.1), 'epochs': 150},
]

for i, arch in enumerate(architectures):
    exp_name = f"nn_arch{i+1}"
    result = run_experiment(
        X_stacked, y_encoded, y_original, species_classes, model_names,
        all_indices,
        nn_config=arch,
        experiment_name=exp_name
    )
    results.append(result)
    print(f"  {exp_name:40s} acc: {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")


phase 2: hyperparameter tuning (all 5 models)

[phase2] testing random forest hyperparameters...
  rf_n100_d10                              acc: 0.9893 ± 0.0114
  rf_n100_d15                              acc: 0.9893 ± 0.0114
  rf_n100_d20                              acc: 0.9877 ± 0.0124
  rf_n200_d10                              acc: 0.9893 ± 0.0114
  rf_n200_d15                              acc: 0.9893 ± 0.0114
  rf_n200_d20                              acc: 0.9893 ± 0.0114
  rf_n300_d10                              acc: 0.9908 ± 0.0089
  rf_n300_d15                              acc: 0.9877 ± 0.0104
  rf_n300_d20                              acc: 0.9893 ± 0.0114


## results aggregation

In [ ]:
print(f"\n{'='*80}")
print("experiment results")
print(f"{'='*80}\n")

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('mean_accuracy', ascending=False)

print(f"total experiments run: {len(df_results)}")
print(f"\ntop 15 configurations:\n")
print(df_results[['experiment_name', 'models', 'n_features', 'mean_accuracy', 'std_accuracy']].head(15).to_string(index=False))

# save detailed results
results_file = f"{config.results_dir}/experiment_results.csv"
df_results.to_csv(results_file, index=False)
print(f"\n[save] detailed results saved to {results_file}")

# save summary
summary = {
    'total_experiments': len(df_results),
    'best_configuration': df_results.iloc[0].to_dict(),
    'top_10_configurations': df_results.head(10)[['experiment_name', 'models', 'mean_accuracy', 'std_accuracy']].to_dict('records'),
    'best_model_combination': df_results[df_results['experiment_name'].str.startswith('combo')].iloc[0].to_dict() if any(df_results['experiment_name'].str.startswith('combo')) else None,
    'efficiency_analysis': {
        '2_models': float(df_results[df_results['n_models'] == 2]['mean_accuracy'].max()) if any(df_results['n_models'] == 2) else None,
        '3_models': float(df_results[df_results['n_models'] == 3]['mean_accuracy'].max()) if any(df_results['n_models'] == 3) else None,
        '4_models': float(df_results[df_results['n_models'] == 4]['mean_accuracy'].max()) if any(df_results['n_models'] == 4) else None,
        '5_models': float(df_results[df_results['n_models'] == 5]['mean_accuracy'].max()) if any(df_results['n_models'] == 5) else None,
    }
}

summary_file = f"{config.results_dir}/experiment_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"[save] summary saved to {summary_file}")


experiment results

total experiments run: 48

top 15 configurations:

             experiment_name                 models  n_features  mean_accuracy  std_accuracy
                    nn_arch1 rf+rb+stats+resnet+cnn         285       0.990816      0.011216
                    nn_arch2 rf+rb+stats+resnet+cnn         285       0.990816      0.012226
              xgb_d10_eta0.1 rf+rb+stats+resnet+cnn         285       0.990816      0.012226
combo_rf+rb+stats+resnet+cnn rf+rb+stats+resnet+cnn         285       0.990816      0.011216
                 rf_n300_d10 rf+rb+stats+resnet+cnn         285       0.990804      0.008916
              xgb_d10_eta0.2 rf+rb+stats+resnet+cnn         285       0.990793      0.007518
               xgb_d5_eta0.2 rf+rb+stats+resnet+cnn         285       0.990793      0.007518
             combo_rf+rb+cnn              rf+rb+cnn         171       0.989278      0.011431
       combo_rf+rb+stats+cnn        rf+rb+stats+cnn         228       0.989278      0.01143

## train final production models

use the best configuration found above to train on all data

In [ ]:
# get best configuration
best_result = df_results.iloc[0]
print(f"\nbest configuration found:")
print(f"  name: {best_result['experiment_name']}")
print(f"  models: {best_result['models']}")
print(f"  accuracy: {best_result['mean_accuracy']:.4f} ± {best_result['std_accuracy']:.4f}")

# extract best config details
best_name = best_result['experiment_name']

# determine which models were used
if best_name.startswith('combo'):
    selected_models = best_result['models'].split('+')
    model_indices = [list(model_names).index(m) for m in selected_models]
    print(f"  selected models: {selected_models}")
else:
    model_indices = list(range(num_models))


best configuration found:
  name: nn_arch1
  models: rf+rb+stats+resnet+cnn
  accuracy: 0.9908 ± 0.0112


In [ ]:
# build feature indices for best model selection
num_classes = len(species_classes)
feature_indices = []
for idx in model_indices:
    start = idx * num_classes
    end = (idx + 1) * num_classes
    feature_indices.extend(range(start, end))

X_best = X_stacked[:, feature_indices]

# create output directory
output_dir = "production"
os.makedirs(output_dir, exist_ok=True)

print(f"\ntraining final production models on all data...")
print(f"  samples: {len(X_best)}")
print(f"  features: {X_best.shape[1]}")


training final production models on all data...
  samples: 651
  features: 285


In [ ]:
# train xgboost meta-learner on all data
print(f"\n[train] training xgboost meta-learner...")
xgb_meta = XGBMetaLearner(max_depth=7, eta=0.1, subsample=0.8, num_boost_round=100)

if xgb_meta.fit(X_best, y_encoded):
    # save model
    xgb_path = f"{output_dir}/final_xgb_meta.pkl"
    with open(xgb_path, 'wb') as f:
        pickle.dump({
            'model': xgb_meta.model,
            'label_encoder': xgb_meta.le,
            'config': {
                'max_depth': 7,
                'eta': 0.1,
                'subsample': 0.8,
                'num_boost_round': 100
            }
        }, f)
    
    print(f"[save] xgboost model saved to {xgb_path}")
else:
    print("[WARNING] failed to train xgboost")


[train] training xgboost meta-learner...
[save] xgboost model saved to production/final_xgb_meta.pkl


In [ ]:
# train neural network meta-learner on all data
print(f"\n[train] training neural network meta-learner...")
nn_meta = NNMetaLearner(X_best.shape[1], num_classes)

if nn_meta.fit(X_best, y_encoded, epochs=100):
    # save model
    nn_path = f"{output_dir}/final_nn_meta.pth"
    torch.save({
        'model_state_dict': nn_meta.model.state_dict(),
        'label_encoder_classes': nn_meta.le.classes_,
        'input_dim': X_best.shape[1],
        'num_classes': num_classes,
        'config': {
            'hidden_dims': (256, 128, 64),
            'dropout_rates': (0.4, 0.3, 0.2),
            'epochs': 100
        }
    }, nn_path)
    
    print(f"[save] neural network model saved to {nn_path}")
else:
    print("[WARNING] failed to train neural network")


[train] training neural network meta-learner...
[save] neural network model saved to production/final_nn_meta.pth


In [ ]:
# save metadata
metadata = {
    'best_configuration': best_result['experiment_name'],
    'best_accuracy': float(best_result['mean_accuracy']),
    'best_accuracy_std': float(best_result['std_accuracy']),
    'selected_models': [model_names[i] for i in model_indices],
    'n_samples': len(X_best),
    'n_classes': num_classes,
    'n_features': X_best.shape[1],
    'species_classes': list(species_classes),
    'seed': config.seed,
}

metadata_path = f"{output_dir}/model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"\n[save] metadata saved to {metadata_path}")

# print summary
print(f"\n{'='*80}")
print("final production models trained")
print(f"{'='*80}\n")

print(f"best configuration:")
print(f"  name: {best_result['experiment_name']}")
print(f"  accuracy: {best_result['mean_accuracy']:.4f} ± {best_result['std_accuracy']:.4f}")
print(f"  models: {', '.join([model_names[i] for i in model_indices])}")

print(f"\nsaved to production/:")
print(f"  - final_xgb_meta.pkl (best for inference)")
print(f"  - final_nn_meta.pth (alternative)")
print(f"  - model_metadata.json (configuration)")

print(f"\nnext step: run predict.py to make predictions")
print(f"{'='*80}\n")


[save] metadata saved to production/model_metadata.json

final production models trained

best configuration:
  name: nn_arch1
  accuracy: 0.9908 ± 0.0112
  models: rf, rb, stats, resnet, cnn

saved to production/:
  - final_xgb_meta.pkl (best for inference)
  - final_nn_meta.pth (alternative)
  - model_metadata.json (configuration)

next step: run predict.py to make predictions

